In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
#Three-Qubit Gates
#Amitava Chakraborty - 15th Jan 2018
from qiskit_backend_config import IBMQ
# Loading your IBM Q account(s)
IBMQ.load_account()

In [ ]:
# Useful additional packages
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from math import pi
# Importing standard Qiskit libraries and configuring account
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_backend_config import execute
from qiskit.visualization import circuit_drawer
from qiskit.circuit.library import HGate, RZGate, CXGate, CCXGate, XGate, SwapGate

from qiskit.quantum_info import state_fidelity
from qiskit_backend_config import BasicAer

backend = BasicAer.get_backend('unitary_simulator')
q_backend = BasicAer.get_backend('qasm_simulator')

In [ ]:
#There are two commonly-used three-qubit gates. 
#Qiskit uses a representation in which the first qubit is on the right-most side of the tensor product
#and the third qubit is on the left-most side

#Toffoli gate (CCX gate) - CONTROL-CONTROL-NOT
#The Toffoli gate flips the third qubit if the first two qubits (LSB) are both 1
#i.e., if a = 1 and b = 1, then flip c
q = QuantumRegister(3)
qc = QuantumCircuit(q)
qc.ccx(q[0], q[1], q[2])
qc.draw(output='mpl')

In [ ]:
job = execute(qc, backend)
job.result().get_unitary(qc, decimals=3)

In [ ]:
# Multi-control Toffoli gate with clean ancillas
num_controls=3
q_controls = QuantumRegister(num_controls)
q_target = QuantumRegister(1)
qc = QuantumCircuit(q_controls, q_target)

if num_controls > 2:
    num_ancillas = num_controls - 2
    q_ancillas = QuantumRegister(num_controls)
    qc.add_register(q_ancillas)
else:
    num_ancillas = 0
    q_ancillas = None

# apply hadamard on control qubits and toffoli gate
qc.mcx(q_controls, q_target[0], q_ancillas, mode='basic')
qc.draw('mpl')

In [ ]:
# execute the circuit and obtain statevector result
backend = BasicAer.get_backend('unitary_simulator')
simulated = execute(qc, backend).result().get_unitary(qc)
print(simulated)

In [ ]:
 # Multi-control Toffoli gate with dirty ancillas
num_controls=6
q_controls = QuantumRegister(num_controls)
q_target = QuantumRegister(1)
qc = QuantumCircuit(q_controls, q_target)

q_ancillas = None
if num_controls <= 4:
    num_ancillas = 0
else:
    num_ancillas = 1
    q_ancillas = QuantumRegister(num_ancillas)
    qc.add_register(q_ancillas)

qc.mcx(q_controls, q_target[0], q_ancillas, mode='advanced')
qc.draw('mpl')

In [ ]:
# execute the circuit and obtain statevector result
backend = BasicAer.get_backend('unitary_simulator')
simulated = execute(qc, backend).result().get_unitary(qc)
print(simulated)

In [ ]:
q_init = QuantumRegister(3)
qc_init = QuantumCircuit(q_init)
x=XGate()
qc_init=qc_init.compose(x,[0]).compose(x,[1]).compose(x,[2])
qc_init.draw('mpl')

In [ ]:
c_measure = ClassicalRegister(2)
qc.add_register(c_measure)
qc = qc.compose(qc_init, qubits=[q_controls[0], q_controls[1], q_controls[2]], inplace=False)
qc.draw('mpl')

In [ ]:
qc = qc.reverse_ops()
qc.measure([q_target[0],q_ancillas[0]],[c_measure[0],c_measure[1]])
qc.draw('mpl')

In [ ]:
job = execute(qc, q_backend, shots=15)
print(job.result().get_counts(qc))

In [ ]:
#Controlled swap gate (Fredkin Gate)
#The Fredkin gate, or the controlled swap gate, 
#exchanges the second and third qubits if the first qubit (LSB) is 1
#i.e., if control qubit == 1, then the 2 other qubits states are swaped
qc = QuantumCircuit(q)
#qc.cswap(q[0], q[1], q[2])
qc.cswap(q[1], q[0], q[2])

qc.draw(output='mpl')

In [ ]:
job = execute(qc, backend)
job.result().get_unitary(qc, decimals=3)

# 3-qubit controlled X gate
ref : https://arxiv.org/pdf/quant-ph/9503016.pdf

Parameters
angle (float) – similar to controlled-U1 gates. 
An angle of π/4 yields the 3-qubit controlled X gate, an angle of π/8 the 3-qubit controlled sqrt(X) gate.


In [ ]:
from qiskit.circuit.library import *
qr = QuantumRegister(4)
qc = QuantumCircuit(qr)
qc.h(qr[0])
from qiskit.circuit import CircuitInstruction
qc.data[0] = CircuitInstruction(C3XGate(), [qr[0], qr[1], qr[2], qr[3]], [])
#qc.data.append((C3XGate(pi/4), [qr[0], qr[1], qr[2], qr[3]], []))
qc.draw('mpl')

In [ ]:
qc = QuantumCircuit(4)
qc.h(1)
cgate = XGate().control(3, ctrl_state=6)
qc.append(cgate, [0, 1, 2, 3])
qc.draw('mpl', scale=1.5)

In [ ]:
# import qiskit.tools.jupyter  # qiskit.tools.jupyter was removed in Qiskit 1.0
circuit = XOR(5, seed=42)
circuit.draw('mpl')  # %circuit_library_info was removed with qiskit.tools.jupyter

In [ ]:
from qiskit import QuantumCircuit
#from qiskit.extensions import HGate
hgate = HGate()
circ = QuantumCircuit(4)
circ.append(hgate.control(3), [0, 1, 2, 3])
print(circ)